# AI Research Assistant Demonstration

This notebook demonstrates the end-to-end functionality of the AI Research Assistant, showcasing:
1. **Document Ingestion**: Parsing text/URLs and indexing into Pinecone using **Google text-embedding-004** (768-dimensional vector space).
2. **Semantic Retrieval**: Querying Pinecone and reranking using the local ms-marco-MiniLM Cross-Encoder.
3. **Orchestrator Synthesis**: Synthesizing answers via the LangChain ReAct Agent powered by **Google Gemini 1.5 Pro**.

In [ ]:
# Step 0: Set up paths and environment variables
import os
import sys
from dotenv import load_dotenv

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Load keys from .env file
load_dotenv(dotenv_path=os.path.join(project_root, '.env'))
print("Environment keys loaded. Ready for execution.")

In [ ]:
# Step 1: Ingest research material
from app.core.ingestion import DocumentIngester

ingester = DocumentIngester()

# We will ingest a public paper abstract about AlphaFold 3
sample_text = """
AlphaFold 3 predicts the structure and interactions of proteins, nucleic acids, small molecules, ions, and chemical modifications.
By employing a diffusion-based architecture rather than standard Multiple Sequence Alignments (MSAs), it predicts coordinates
directly for all atoms in a complex. This results in a 50% improvement in predicting protein-ligand structural interactions
compared to classical molecular docking techniques. It sets a new standard for biological complex simulation.
"""

doc = ingester.load_text(sample_text, source_name="alphafold3_nature_abstract")
chunks = ingester.ingest_document(doc)

print(f"Ingestion completed. Created {len(chunks)} chunk(s).")
print(f"First chunk excerpt: {chunks[0]['text'][:120]}...")

In [ ]:
# Step 2: Test the Retrieval + Reranking Pipeline
from app.core.retrieval import DocumentRetriever

retriever = DocumentRetriever()
query = "How does AlphaFold 3 predict structures and what are its accuracy improvements?"

# Retrieve top 3 reranked chunks
retrieved_chunks = retriever.retrieve(query, top_k=8, final_top_k=3)

print(f"Retrieved {len(retrieved_chunks)} relevant chunk(s):")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\n[{i+1}] Source: {chunk['metadata'].get('source')} | Rerank Score: {chunk.get('rerank_score'):.4f}")
    print(f"Text excerpt: {chunk['text'][:150]}...")

In [ ]:
# Step 3: Run the Reasoning Agent with Citations
from app.core.agent import ResearchReActAgent

agent = ResearchReActAgent()
query = "What are the latest breakthroughs in protein folding prediction and how do they compare to AlphaFold2?"

response = agent.run(query)

print("=== Synthesized Response ===")
print(response["answer"])
print("\n=== Source Grounding Confidence ===")
print(f"Confidence Score: {response['confidence_score'] * 100:.1f}%")
print("\n=== Citations ===")
for cit in response["citations"]:
    print(f"[{cit['index']}] {cit['title']} ({cit['source']})")